# 📦 Optimisation Multi-Étages de la Valeur de Stock — MILP Pyomo

## Contexte
Réduction de la valeur de stock du site (MP + SF + PF) de **11 M€** vers une cible de **9 M€** fin septembre, en optimisant les décisions de production et d'approvisionnement sur un horizon de 4 mois (juin → septembre).

## Architecture
- **Couche 1 — EOQ** : calibration des paramètres économiques (stock de sécurité).
- **Couche 2 — MILP multi-périodes, multi-étages** : optimisation de la trajectoire de stock sous contraintes de bilan matière, capacité, taux de service et cible fiscale.


---
## 0. Imports

In [ ]:
import os, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100
from pyomo.environ import *


---
## 1. Définition du modèle abstrait Pyomo

In [ ]:
model = AbstractModel()

# ── Ensembles ─────────────────────────────────────────────
model.MP = Set()
model.SF = Set()
model.PF = Set()
model.T  = Param(within=PositiveIntegers)
model.PER = RangeSet(1, model.T)
model.AoutSet  = Set(within=model.PER)
model.t_juillet = Param(within=PositiveIntegers)

# ── Paramètres économiques ────────────────────────────────
model.TARGET  = Param()
model.tau     = Param()
model.z_alpha = Param()
model.M       = Param()
model.V_max   = Param()

# ── Par référence MP ──────────────────────────────────────
model.prix_MP       = Param(model.MP)
model.stock_init_MP = Param(model.MP)
model.LT            = Param(model.MP)
model.MOQ           = Param(model.MP)
model.volume_MP     = Param(model.MP)
model.SS_MP         = Param(model.MP)

# ── Par référence SF ──────────────────────────────────────
model.prix_SF       = Param(model.SF)
model.stock_init_SF = Param(model.SF)
model.ct_SF         = Param(model.SF)
model.MLS_SF        = Param(model.SF)
model.charge_SF     = Param(model.SF)
model.volume_SF     = Param(model.SF)
model.SS_SF         = Param(model.SF)

# ── Par référence PF ──────────────────────────────────────
model.prix_PF       = Param(model.PF)
model.stock_init_PF = Param(model.PF)
model.ct_PF         = Param(model.PF)
model.MLS_PF        = Param(model.PF)
model.charge_PF     = Param(model.PF)
model.volume_PF     = Param(model.PF)
model.SS_PF         = Param(model.PF)

# ── Flux ──────────────────────────────────────────────────
model.demande = Param(model.PF, model.PER, default=0)
model.OC      = Param(model.MP, model.PER, default=0)
model.WIP_SF  = Param(model.SF, model.PER, default=0)
model.WIP_PF  = Param(model.PF, model.PER, default=0)
model.a       = Param(model.MP, model.SF,  default=0)
model.b       = Param(model.SF, model.PF,  default=0)
model.Cap_SF  = Param(model.PER)
model.Cap_PF  = Param(model.PER)

print("✅  Paramètres abstraits déclarés")


In [ ]:
# ── Variables de décision ──────────────────────────────────
# Stocks en fin de période t (photos de fin de mois)
model.S_MP = Var(model.MP, model.PER, domain=NonNegativeReals)
model.S_SF = Var(model.SF, model.PER, domain=NonNegativeReals)
model.S_PF = Var(model.PF, model.PER, domain=NonNegativeReals)

# Quantités commandées / produites (lancées en début de t)
model.O_MP = Var(model.MP, model.PER, domain=NonNegativeReals)
model.P_SF = Var(model.SF, model.PER, domain=NonNegativeReals)
model.P_PF = Var(model.PF, model.PER, domain=NonNegativeReals)

print("✅  Variables de décision déclarées")


In [ ]:
# ── Fonction objectif : minimiser la valeur totale du stock sur tout l'horizon ──
def obj_rule(m):
    return (
        sum(m.prix_MP[i] * m.S_MP[i,t] for i in m.MP for t in m.PER)
      + sum(m.prix_SF[j] * m.S_SF[j,t] for j in m.SF for t in m.PER)
      + sum(m.prix_PF[l] * m.S_PF[l,t] for l in m.PF for t in m.PER)
    )
model.obj = Objective(rule=obj_rule, sense=minimize)
print("✅  Fonction objectif définie")


In [ ]:
# ── C1 — Bilan stock Matière Première ─────────────────────
# S[i,t] = S[i,t-1] + OC[i,t] + O[i,t-LT_i] - sum_j(a[i,j]*P_SF[j,t])
def c1_rule(m, i, t):
    S_prev    = m.stock_init_MP[i] if t == 1 else m.S_MP[i, t-1]
    t_cmd     = t - int(value(m.LT[i]))
    reception = m.O_MP[i, t_cmd] if t_cmd >= 1 else 0
    conso     = sum(m.a[i,j] * m.P_SF[j,t] for j in m.SF if value(m.a[i,j]) > 0)
    return m.S_MP[i,t] == S_prev + m.OC[i,t] + reception - conso
model.C1 = Constraint(model.MP, model.PER, rule=c1_rule)

# ── C2 — Bilan stock Semi-Fini ────────────────────────────
# S[j,t] = S[j,t-1] + WIP_SF[j,t] + P_SF[j,t-ct_j] - sum_l(b[j,l]*P_PF[l,t])
def c2_rule(m, j, t):
    S_prev    = m.stock_init_SF[j] if t == 1 else m.S_SF[j, t-1]
    t_lanc    = t - int(value(m.ct_SF[j]))
    sortie    = m.P_SF[j, t_lanc] if t_lanc >= 1 else 0
    conso     = sum(m.b[j,l] * m.P_PF[l,t] for l in m.PF if value(m.b[j,l]) > 0)
    return m.S_SF[j,t] == S_prev + m.WIP_SF[j,t] + sortie - conso
model.C2 = Constraint(model.SF, model.PER, rule=c2_rule)

# ── C3 — Bilan stock Produit Fini ─────────────────────────
# S[l,t] = S[l,t-1] + WIP_PF[l,t] + P_PF[l,t-ct_l] - D[l,t]
def c3_rule(m, l, t):
    S_prev = m.stock_init_PF[l] if t == 1 else m.S_PF[l, t-1]
    t_lanc = t - int(value(m.ct_PF[l]))
    sortie = m.P_PF[l, t_lanc] if t_lanc >= 1 else 0
    return m.S_PF[l,t] == S_prev + m.WIP_PF[l,t] + sortie - m.demande[l,t]
model.C3 = Constraint(model.PF, model.PER, rule=c3_rule)

# ── C5 — Cible fiscale fin d'horizon ──────────────────────
def c5_rule(m):
    T_fin = int(value(m.T))
    return (
        sum(m.prix_MP[i] * m.S_MP[i, T_fin] for i in m.MP)
      + sum(m.prix_SF[j] * m.S_SF[j, T_fin] for j in m.SF)
      + sum(m.prix_PF[l] * m.S_PF[l, T_fin] for l in m.PF)
      <= m.TARGET
    )
model.C5 = Constraint(rule=c5_rule)

# ── C6 — Taux de service minimum PF ──────────────────────
def c6_rule(m, l, t):
    return m.S_PF[l,t] >= m.SS_PF[l]
model.C6 = Constraint(model.PF, model.PER, rule=c6_rule)

# ── C7 — Capacité de production ───────────────────────────
def c7_sf_rule(m, t):
    return sum(m.charge_SF[j] * m.P_SF[j,t] for j in m.SF) <= m.Cap_SF[t]
model.C7_SF = Constraint(model.PER, rule=c7_sf_rule)

def c7_pf_rule(m, t):
    return sum(m.charge_PF[l] * m.P_PF[l,t] for l in m.PF) <= m.Cap_PF[t]
model.C7_PF = Constraint(model.PER, rule=c7_pf_rule)

# ── C8 — Capacité de stockage globale ─────────────────────
def c8_rule(m, t):
    return (
        sum(m.volume_MP[i] * m.S_MP[i,t] for i in m.MP)
      + sum(m.volume_SF[j] * m.S_SF[j,t] for j in m.SF)
      + sum(m.volume_PF[l] * m.S_PF[l,t] for l in m.PF)
      <= m.V_max
    )
model.C8 = Constraint(model.PER, rule=c8_rule)

print("✅  Contraintes C1→C8 définies")


---
## 2. Fonctions de génération logique (section 5)

In [ ]:
def calc_eoq(D, C, h):
    h_s = max(h, 0.01)
    v   = 2 * D * C / h_s
    return math.sqrt(v) if (v > 0 and math.isfinite(v)) else 0.0

def norm(x): return str(x).strip().upper()

def generer_params_logiques(prix_MP, prix_SF, prix_PF, T, t_aout):
    ref_MP = max(max(prix_MP.values()) if prix_MP else 50, 0.01)
    ref_SF = max(max(prix_SF.values()) if prix_SF else 200, 0.01)
    ref_PF = max(max(prix_PF.values()) if prix_PF else 800, 0.01)
    volume_MP = {i: round(max(0.001, 0.003*(p/ref_MP)**0.4), 4) for i,p in prix_MP.items()}
    volume_SF = {j: round(max(0.002, 0.025*(p/ref_SF)**0.5), 4) for j,p in prix_SF.items()}
    volume_PF = {l: round(max(0.005, 0.070*(p/ref_PF)**0.5), 4) for l,p in prix_PF.items()}
    charge_SF = {j: round(max(0.5, 1.0 + max(p,0.01)/300), 2) for j,p in prix_SF.items()}
    charge_PF = {l: round(max(1.0, 2.0 + max(p,0.01)/400), 2) for l,p in prix_PF.items()}
    ct_SF  = {j: 1 for j in prix_SF}   # 1 mois pour tous
    ct_PF  = {l: 1 for l in prix_PF}   # 1 mois pour tous
    base_SF = round(2*8*22*0.85)        # 300 h/mois
    base_PF = round(1*8*22*0.85)        # 150 h/mois
    Cap_SF = {t: round(base_SF*0.35) if t in t_aout else base_SF for t in T}
    Cap_PF = {t: round(base_PF*0.35) if t in t_aout else base_PF for t in T}
    return volume_MP, volume_SF, volume_PF, charge_SF, charge_PF, ct_SF, ct_PF, Cap_SF, Cap_PF

print("✅  Fonctions de génération définies")


---
## 3. Chargement des données (`METTRE EN PLACE LA DATA`)

In [ ]:
DOSSIER_DATA  = r"/content/sample_data"   # ◄ adapter si nécessaire
def f(nom): return os.path.join(DOSSIER_DATA, nom)

T             = [1, 2, 3, 4]
T_max         = 4
t_juillet     = 2
t_aout        = [3]
noms_periodes = {1:"Juin", 2:"Juillet", 3:"Août", 4:"Septembre"}

# ── Facteur d'échelle : ramène la valeur totale du stock à 11 000 000 € ────────
# Valeur réelle du stock = 4 813 161 €  →  facteur = 11 000 000 / 4 813 161
SCALE_PRIX = 2.285400
TARGET      = 9000000

print(f"Facteur d'échelle appliqué aux prix : {SCALE_PRIX:.4f}")
print(f"Objectif    : stock initial ≈ 11 000 000 €  |  cible fin septembre = {TARGET:,.0f} €")


In [ ]:
# ══════════════════════════════════════════════════════
# 3.1 — Stock actuel (PN__STOCK__PRIX_EN_STOCK_TYPE_PN.XLSX)
# ══════════════════════════════════════════════════════
df_s = pd.read_excel(f("PN__STOCK__PRIX_EN_STOCK_TYPE_PN.XLSX"))
df_s.columns = df_s.columns.str.strip()
df_s = df_s[df_s['Material Type'].isin(['ZROH','ZHLB','ZFRT'])].copy()
df_s['Material'] = df_s['Material'].apply(norm)
df_s = df_s.drop_duplicates('Material').set_index('Material')

for col in ['Unrestricted Stock','Prix_Unitaire','Inv Value in EUR']:
    df_s[col] = pd.to_numeric(df_s[col], errors='coerce').fillna(0)

# Application du facteur d'échelle sur le prix
df_s['prix_scaled'] = df_s['Prix_Unitaire'] * SCALE_PRIX

I = df_s[df_s['Material Type']=='ZROH'].index.tolist()
J = df_s[df_s['Material Type']=='ZHLB'].index.tolist()
L = df_s[df_s['Material Type']=='ZFRT'].index.tolist()

prix_MP       = df_s.loc[I,'prix_scaled'].to_dict()
stock_init_MP = df_s.loc[I,'Unrestricted Stock'].to_dict()
prix_SF       = df_s.loc[J,'prix_scaled'].to_dict()
stock_init_SF = df_s.loc[J,'Unrestricted Stock'].to_dict()
prix_PF       = df_s.loc[L,'prix_scaled'].to_dict()
stock_init_PF = df_s.loc[L,'Unrestricted Stock'].to_dict()
type_par_pn   = df_s['Material Type'].to_dict()

val_MP  = sum(prix_MP[i]*stock_init_MP[i] for i in I)
val_SF  = sum(prix_SF[j]*stock_init_SF[j] for j in J)
val_PF  = sum(prix_PF[l]*stock_init_PF[l] for l in L)
val_tot = val_MP + val_SF + val_PF

print("=" * 62)
print("  STOCK INITIAL (après mise à l'échelle)")
print("=" * 62)
print(f"  MP  : {len(I):4d} références  →  {val_MP:>14,.0f}  €")
print(f"  SF  : {len(J):4d} références  →  {val_SF:>14,.0f}  €")
print(f"  PF  : {len(L):4d} références  →  {val_PF:>14,.0f}  €")
print(f"  {'─'*46}")
print(f"  TOTAL                      →  {val_tot:>14,.0f}  €")
print(f"  Cible TARGET               →  {TARGET:>14,.0f}  €")
print(f"  Réduction nécessaire       →  {val_tot-TARGET:>14,.0f}  €")
print("=" * 62)


In [ ]:
# ══════════════════════════════════════════════════════
# 3.2 — Lead Times (LEAD_TIME_FOURNISSEUR_AVEC_PN.xlsx)
# Lead time capé à T_max = 4 mois : au-delà, la commande
# n'arriverait pas dans l'horizon → on plafonne à 4.
# ══════════════════════════════════════════════════════
df_lt = pd.read_excel(f("LEAD_TIME_FOURNISSEUR_AVEC_PN.xlsx"))
df_lt.columns = df_lt.columns.str.strip()
col_lt = [c for c in df_lt.columns if 'LEAD' in c.upper()][0]
df_lt['PN'] = df_lt['PN'].apply(norm)

lt_jours = df_lt.groupby('PN')[col_lt].max().to_dict()

LT = {}
for i in I:
    j = lt_jours.get(i, 120)              # 120j par défaut si manquant
    mois = max(1, math.ceil(float(j)/30))
    LT[i] = min(mois, T_max)              # ← cap à T_max (4 mois)

print(f"✅  Lead Times chargés — {len(LT)} MP")
print(f"   Distribution LT (mois, après cap à {T_max}) :")
from collections import Counter
print("  ", dict(Counter(LT.values())))
print(f"   → LT=4 signifie 'commande n'arrive qu'en t=T_max au plus tôt'")


In [ ]:
# ══════════════════════════════════════════════════════
# 3.3 — MOQ / MLS (PN_MLS.xls)
# ══════════════════════════════════════════════════════
try:
    df_moq = pd.read_excel(f("PN_MLS.xls"), engine='xlrd')
    df_moq.columns = df_moq.columns.str.strip()
    df_moq['Material'] = df_moq['Material'].apply(norm)
    moq_s = df_moq.set_index('Material')['Cur. Minimum lot size']
    def get_moq(pn, defaut=1.0):
        v = moq_s.get(pn, None)
        return float(v) if (v is not None and not (isinstance(v,float) and math.isnan(v))) else defaut
    MOQ    = {i: get_moq(i) for i in I}
    MLS_SF = {j: get_moq(j) for j in J}
    MLS_PF = {l: get_moq(l) for l in L}
    print(f"✅  MOQ/MLS chargés — MP:{len(MOQ)}  SF:{len(MLS_SF)}  PF:{len(MLS_PF)}")
except Exception as e:
    MOQ    = {i: 1.0 for i in I}
    MLS_SF = {j: 1.0 for j in J}
    MLS_PF = {l: 1.0 for l in L}
    print(f"⚠️  MOQ/MLS : {e} → défaut = 1 appliqué")


In [ ]:
# ══════════════════════════════════════════════════════
# 3.4 — Demande client (DEMAND_ANALYSSIS.xlsx)
# Colonnes semaines FY TE : 2026/36→t=1 Juin
#                           2026/40→t=2 Juillet
#                           2026/44→t=3 Août
#                           2026/48→t=4 Septembre
# (1 mois ≈ 4 semaines fiscales TE, FY commence en octobre)
# ══════════════════════════════════════════════════════
week_to_t = {}
for w in range(36,40): week_to_t[f"2026/{w}"] = 1
for w in range(40,44): week_to_t[f"2026/{w}"] = 2
for w in range(44,48): week_to_t[f"2026/{w}"] = 3
for w in range(48,52): week_to_t[f"2026/{w}"] = 4

df_dem = pd.read_excel(f("DEMAND_ANALYSSIS.xlsx"))
df_dem.columns = [str(c).strip() for c in df_dem.columns]
df_dem['MATERIAL'] = df_dem['MATERIAL'].apply(norm)
df_dem = df_dem[df_dem['MATERIAL'].isin(L)].copy()

week_ok = [w for w in week_to_t if w in df_dem.columns]
for w in week_ok:
    df_dem[w] = pd.to_numeric(df_dem[w], errors='coerce').fillna(0)

df_dem = df_dem.set_index('MATERIAL')

demande = {}
for l in L:
    for t in T:
        cols_t = [w for w in week_ok if week_to_t[w]==t]
        if l in df_dem.index and cols_t:
            demande[(l,t)] = max(0, int(df_dem.loc[l, cols_t].sum()))
        else:
            demande[(l,t)] = 0

n_pf_dem = sum(1 for l in L if sum(demande[(l,t)] for t in T)>0)
tot_dem  = sum(demande.values())
print(f"✅  Demande chargée — {len(week_ok)} semaines → 4 mois")
print(f"   PF avec demande > 0 : {n_pf_dem} / {len(L)}")
print(f"   Demande totale horizon : {tot_dem:,.0f} unités")

df_dem_agg = pd.DataFrame(
    {noms_periodes[t]: {l: demande[(l,t)] for l in L if sum(demande[(l,tt)] for tt in T)>0} for t in T}
).head(10)
print("\n  Extrait demande mensuelle (10 premiers PF avec demande > 0) :")
display(df_dem_agg)


In [ ]:
# ══════════════════════════════════════════════════════
# 3.5 — Pipeline SAP : WIP et achats en cours
# ══════════════════════════════════════════════════════
OC     = {i: {t: 0.0 for t in T} for i in I}
WIP_SF = {j: {t: 0.0 for t in T} for j in J}
WIP_PF = {l: {t: 0.0 for t in T} for l in L}

def charger_pipeline(sheet, col_pn, col_qte, dico, liste):
    try:
        df = pd.read_excel(f("WIP_Achats_En_Cours.xlsx"), sheet_name=sheet)
        df.columns = [str(c).strip() for c in df.columns]
        df[col_pn] = df[col_pn].apply(norm)
        for _, row in df.iterrows():
            pn = row[col_pn]; t = int(row.get('Mois Modèle (t)', 0))
            q  = float(row.get(col_qte, 0) or 0)
            if pn in dico and t in T: dico[pn][t] += q
        print(f"   ✅  {sheet} : {len(df)} lignes")
    except Exception as e:
        print(f"   ⚠️  {sheet} : {e}")

print("Chargement pipeline SAP :")
charger_pipeline("Achats_En_Cours_MP",  "PN",    "Qté Commandée", OC,     I)
charger_pipeline("WIP_Semi_Finis",      "PN SF", "Qté En-Cours",  WIP_SF, J)
charger_pipeline("WIP_Produits_Finis",  "PN PF", "Qté En-Cours",  WIP_PF, L)

val_oc  = sum(prix_MP[i]*OC[i][t] for i in I for t in T)
val_wsf = sum(prix_SF[j]*WIP_SF[j][t] for j in J for t in T)
val_wpf = sum(prix_PF[l]*WIP_PF[l][t] for l in L for t in T)
print(f"\n   Valeur OC  en transit : {val_oc:>12,.0f} €")
print(f"   Valeur WIP SF         : {val_wsf:>12,.0f} €")
print(f"   Valeur WIP PF         : {val_wpf:>12,.0f} €")


In [ ]:
# ══════════════════════════════════════════════════════
# 3.6 — Nomenclature BOM (ZBOM.XLSX)
# qty_unitaire = Quantity / Base_quantity
# ZROH → ZHLB : matrice a (MP → SF)
# ZHLB → ZFRT : matrice b (SF → PF)
# ══════════════════════════════════════════════════════
df_bom = pd.read_excel(f("ZBOM.XLSX"),
    usecols=['Parent','Base quantity','Material / Doc','Quantity','Material Type'])
df_bom.columns = df_bom.columns.str.strip()
df_bom['Parent']         = df_bom['Parent'].apply(norm)
df_bom['Material / Doc'] = df_bom['Material / Doc'].apply(norm)
df_bom['Base quantity']  = pd.to_numeric(df_bom['Base quantity'], errors='coerce').fillna(1)
df_bom['Quantity']       = pd.to_numeric(df_bom['Quantity'],       errors='coerce').fillna(0)
df_bom['qty_u'] = df_bom['Quantity'] / df_bom['Base quantity'].replace(0,1)

a_bom = {i: {j: 0.0 for j in J} for i in I}
b_bom = {j: {l: 0.0 for l in L} for j in J}
n_a = n_b = n_ign = 0

for _, row in df_bom.iterrows():
    parent = row['Parent']; comp = row['Material / Doc']; qty = float(row['qty_u'])
    tp = type_par_pn.get(parent,''); tc = type_par_pn.get(comp,'')
    if tc=='ZROH' and tp=='ZHLB' and comp in a_bom and parent in a_bom[comp]:
        a_bom[comp][parent] += qty; n_a += 1
    elif tc=='ZHLB' and tp=='ZFRT' and comp in b_bom and parent in b_bom[comp]:
        b_bom[comp][parent] += qty; n_b += 1
    else:
        n_ign += 1

print(f"✅  BOM chargée — {len(df_bom)} lignes")
print(f"   MP→SF (a) : {n_a} liens  |  SF→PF (b) : {n_b} liens  |  Ignorés : {n_ign}")


In [ ]:
# ══════════════════════════════════════════════════════
# 3.7 — Génération des données complémentaires
# ══════════════════════════════════════════════════════
(volume_MP, volume_SF, volume_PF,
 charge_SF, charge_PF,
 ct_SF, ct_PF,
 Cap_SF, Cap_PF) = generer_params_logiques(prix_MP, prix_SF, prix_PF, T, t_aout)

# ── Stocks de sécurité SS (EOQ) ───────────────────────────
# Demandes annuelles par cascade BOM
D_an_PF = {l: max(1, sum(demande.get((l,t),0) for t in T)*3) for l in L}
D_an_SF = {j: max(1, sum(b_bom[j][l]*D_an_PF[l] for l in L)) for j in J}
D_an_MP = {i: max(1, sum(a_bom[i][j]*D_an_SF[j] for j in J)) for i in I}

tau     = 0.20; z_alpha = 1.65
SS_MP   = {}; SS_SF = {}; SS_PF = {}

for i in I:
    h  = tau * max(prix_MP[i], 0.01)
    Qs = calc_eoq(D_an_MP[i], max(100, prix_MP[i]*2.5), h)
    SS_MP[i] = z_alpha * max(5, round(stock_init_MP[i]*0.08)) * math.sqrt(max(LT[i],1))

for j in J:
    h  = tau * max(prix_SF[j], 0.01)
    Qs = calc_eoq(D_an_SF[j], max(150, prix_SF[j]*1.5), h)
    SS_SF[j] = z_alpha * max(5, round(stock_init_SF[j]*0.08)) * math.sqrt(max(ct_SF[j],1))

for l in L:
    h  = tau * max(prix_PF[l], 0.01)
    Qs = calc_eoq(D_an_PF[l], max(200, prix_PF[l]*1.0), h)
    dem_moy = sum(demande.get((l,t),0) for t in T)/len(T)
    # SS = 0 si pas de demande (stock de sécurité non contraignant pour les PF sans demande)
    SS_PF[l] = 0.0 if dem_moy == 0 else z_alpha * max(5, round(dem_moy*0.12)) * math.sqrt(max(ct_PF[l],1))

print("✅  Données complémentaires générées")
print(f"   Cap SF : {Cap_SF}  |  Cap PF : {Cap_PF}")


In [ ]:
# ══════════════════════════════════════════════════════
# 3.8 — Construction du DataPortal Pyomo
# Note : scalaires → {None: valeur}  |  dicts indexés → directement
# ══════════════════════════════════════════════════════
data = DataPortal()

data['T']          = {None: T_max}
data['t_juillet']  = {None: t_juillet}
data['AoutSet']    = t_aout
data['MP'] = I; data['SF'] = J; data['PF'] = L

data['TARGET']    = {None: TARGET}
data['tau']       = {None: tau}
data['z_alpha']   = {None: z_alpha}
data['M']         = {None: 1e9}
data['V_max']     = {None: 1e12}   # pas de contrainte stockage

data['prix_MP']       = prix_MP
data['stock_init_MP'] = stock_init_MP
data['LT']            = LT
data['MOQ']           = MOQ
data['volume_MP']     = volume_MP
data['SS_MP']         = SS_MP

data['prix_SF']       = prix_SF
data['stock_init_SF'] = stock_init_SF
data['ct_SF']         = ct_SF
data['MLS_SF']        = MLS_SF
data['charge_SF']     = charge_SF
data['volume_SF']     = volume_SF
data['SS_SF']         = SS_SF

data['prix_PF']       = prix_PF
data['stock_init_PF'] = stock_init_PF
data['ct_PF']         = ct_PF
data['MLS_PF']        = MLS_PF
data['charge_PF']     = charge_PF
data['volume_PF']     = volume_PF
data['SS_PF']         = SS_PF

data['demande'] = {(l,t): demande[(l,t)] for l in L for t in T}
data['OC']      = {(i,t): OC[i][t]        for i in I for t in T}
data['WIP_SF']  = {(j,t): WIP_SF[j][t]    for j in J for t in T}
data['WIP_PF']  = {(l,t): WIP_PF[l][t]    for l in L for t in T}
data['a']       = {(i,j): a_bom[i][j]     for i in I for j in J if a_bom[i][j]>0}
data['b']       = {(j,l): b_bom[j][l]     for j in J for l in L if b_bom[j][l]>0}
data['Cap_SF']  = Cap_SF
data['Cap_PF']  = Cap_PF

instance = model.create_instance(data)

print("=" * 62)
print("  ✅  INSTANCE CONCRÈTE — PRÊTE À RÉSOUDRE")
print("=" * 62)
print(f"  MP  : {len(I):4d}  |  SF : {len(J):4d}  |  PF : {len(L):4d}")
print(f"  Valeur initiale : {val_tot:>14,.0f}  €")
print(f"  Cible TARGET    : {TARGET:>14,.0f}  €")
print(f"  À réduire       : {val_tot-TARGET:>14,.0f}  €")
print(f"  Contraintes     : {sum(1 for _ in instance.component_objects(Constraint, active=True))}")
print("=" * 62)


---
## 4. Résolution

In [ ]:
def resoudre(inst, solveur='cbc', verbose=False):
    opt = SolverFactory(solveur)
    return opt.solve(inst, tee=verbose)

print("Résolution en cours (solver CBC)...")
res = resoudre(instance)
status = str(res.solver.termination_condition)

print()
print("=" * 62)
print("  RÉSULTAT DE LA RÉSOLUTION")
print("=" * 62)
print(f"  Statut solveur  : {status}")
try:
    z_star = value(instance.obj)
    print(f"  Valeur obj Z*   : {z_star:>14,.0f}  €·mois")
except:
    print("  Valeur obj      : indisponible")
print("=" * 62)


In [ ]:
# ── Vérification cible C5 ─────────────────────────────────
T_fin = T_max
val_fin = (
    sum(value(instance.prix_MP[i])*value(instance.S_MP[i,T_fin]) for i in I)
  + sum(value(instance.prix_SF[j])*value(instance.S_SF[j,T_fin]) for j in J)
  + sum(value(instance.prix_PF[l])*value(instance.S_PF[l,T_fin]) for l in L)
)
print(f"  Valeur stock fin {noms_periodes[T_fin]:<9} : {val_fin:>14,.0f}  €")
print(f"  Cible TARGET                  : {TARGET:>14,.0f}  €")
print(f"  Cible respectée               : {'✅ Oui' if val_fin <= TARGET else '❌ Non'}")


In [ ]:
# ── Tableau de trajectoire de stock par étage et par période ─────────────────
valeurs = []
for t in T:
    v_mp = sum(value(instance.prix_MP[i])*value(instance.S_MP[i,t]) for i in I)
    v_sf = sum(value(instance.prix_SF[j])*value(instance.S_SF[j,t]) for j in J)
    v_pf = sum(value(instance.prix_PF[l])*value(instance.S_PF[l,t]) for l in L)
    valeurs.append({"Période":noms_periodes[t], "MP (€)":v_mp, "SF (€)":v_sf,
                    "PF (€)":v_pf, "Total (€)":v_mp+v_sf+v_pf})
df_val = pd.DataFrame(valeurs).set_index("Période")

print("Valeur de stock par période et par étage :")
display(df_val.applymap(lambda x: f"{x:,.0f}"))

# ── Graphique de trajectoire ──────────────────────────────
fig, ax = plt.subplots(figsize=(10,5))
for col, col_eu, color in [("MP (€)","MP","steelblue"),
                             ("SF (€)","SF","darkorange"),
                             ("PF (€)","PF","seagreen")]:
    ax.plot(df_val.index, df_val[col], marker='o', label=col_eu, color=color)
ax.plot(df_val.index, df_val["Total (€)"], marker='s', lw=2.5, color='black', label='Total')
ax.axhline(val_tot,  color='grey',  ls=':',  lw=1.5, label=f"Stock initial ({val_tot:,.0f} €)")
ax.axhline(TARGET,   color='red',   ls='--', lw=2.0, label=f"Cible TARGET ({TARGET:,.0f} €)")
ax.set_title("Trajectoire de la valeur de stock — Juin à Septembre", fontsize=12, fontweight='bold')
ax.set_xlabel("Période"); ax.set_ylabel("Valeur (€)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x/1e6:.1f} M€'))
ax.legend(loc='upper right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ── Top 20 PF : réduction de stock la plus forte ──────────────────────────────
lignes = []
for l in L:
    s0 = prix_PF[l]*stock_init_PF[l]
    sT = value(instance.prix_PF[l])*value(instance.S_PF[l,T_fin])
    lignes.append({"PN (PF)":l, "Stock initial (€)":s0,
                   f"Stock fin {noms_periodes[T_fin]} (€)":sT,
                   "Réduction (€)":s0-sT, "Réduction (%)":((s0-sT)/max(s0,0.01))*100})

df_pf_res = pd.DataFrame(lignes).sort_values("Réduction (€)", ascending=False).head(20)
df_pf_res = df_pf_res.set_index("PN (PF)")
for col in df_pf_res.columns:
    if "€" in col: df_pf_res[col] = df_pf_res[col].map(lambda x: f"{x:,.0f}")
    elif "%" in col: df_pf_res[col] = df_pf_res[col].map(lambda x: f"{x:.1f}%")
print("Top 20 PF — Plus fortes réductions de stock (en valeur) :")
display(df_pf_res)


In [ ]:
# ── Production SF/PF recommandée par le modèle ────────────────────────────────
prod_sf = [(j,t, value(instance.P_SF[j,t])) for j in J for t in T
           if value(instance.P_SF[j,t]) and value(instance.P_SF[j,t]) > 0.1]
prod_pf = [(l,t, value(instance.P_PF[l,t])) for l in L for t in T
           if value(instance.P_PF[l,t]) and value(instance.P_PF[l,t]) > 0.1]

print(f"Production SF recommandée par le modèle : {len(prod_sf)} lancement(s) d'OF")
if prod_sf:
    df_psf = pd.DataFrame(prod_sf, columns=['PN SF','Période (t)','Quantité'])
    df_psf['Période'] = df_psf['Période (t)'].map(noms_periodes)
    display(df_psf[['PN SF','Période','Quantité']].head(15))

print(f"\nProduction PF recommandée par le modèle : {len(prod_pf)} lancement(s) d'OF")
if prod_pf:
    df_ppf = pd.DataFrame(prod_pf, columns=['PN PF','Période (t)','Quantité'])
    df_ppf['Période'] = df_ppf['Période (t)'].map(noms_periodes)
    display(df_ppf[['PN PF','Période','Quantité']].head(15))
